# Prepare SFT for the first LLM

In [3]:
import json
from pathlib import Path
import re
from copy import deepcopy
import random

# Process both directories
json_dir_no_penalty = Path("../generated_data_no_penalty/json_processed_fixed")
json_dir_penalty = Path("../generated_data_penalty/json_processed_fixed")
output_file = "sft_data_llm1.jsonl"

# Find all JSON files from both directories
json_files_no_penalty = sorted(json_dir_no_penalty.glob("*.json"))
json_files_penalty = sorted(json_dir_penalty.glob("*.json"))
json_files = json_files_penalty #json_files_no_penalty + 

random.shuffle(json_files)

print(f"Found {len(json_files_no_penalty)} JSON files in generated_data_no_penalty/json_processed")
print(f"Found {len(json_files_penalty)} JSON files in generated_data_penalty/json_processed")
print(f"Total: {len(json_files)} JSON files")

processed_count = 0
skipped_count = 0
with open(output_file, 'w', encoding='utf-8') as fout:
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as fin:
                data = json.load(fin)
            
            #old_correct = data.get('old_nl_description_correct', False)
            #if old_correct:
            #    skipped_count += 1
            #    continue
            
            # Extract required fields
            nl_description = data.get('nl_description', '')
            pygeox_sft_thinking_temp = data.get('pygeox_sft_thinking', '')
            pygeox_code_temp = data.get('pygeox_code_fixed', '')
            if not pygeox_code_temp:
                pygeox_code_temp = data.get('pygeox_code', '')
            #pygeox_sft_verify = data.get('pygeox_sft_verify', '')
            #replace instances of get_object('x') -> by x again

            pygeox_sft_thinking = re.sub(r"scene.get_object\('([^']+)'\)", r"\1", pygeox_sft_thinking_temp)
            #remove the scene.get_object
            pygeox_code_fixed = re.sub(r"scene.get_object\('([^']+)'\)", r"\1", pygeox_code_temp)
            #add the line_ prefix
            pygeox_code_fixed = re.sub(r'\bline\s+([A-Z]{2,})\b', r'line_\1', pygeox_code_fixed)
            pygeox_code_fixed = re.sub(r"'([a-zA-Z_][a-zA-Z0-9_]*)'\.", r"\1.", pygeox_code_fixed) 
            # Skip if required fields are missing or empty
            if not nl_description or not pygeox_sft_thinking or not pygeox_code_fixed:# or not pygeox_sft_verify:
                skipped_count += 1
                continue
            
            # Construct user message
            user_content = nl_description
            if not user_content.endswith('.'):
                user_content += '.'
            user_content += " Please generate PyGeoX code that represents this diagram."
            
            # Construct assistant message with format: <think>...</think> <answer>...</answer> <verify>...</verify>
            assistant_content = f"```python\n{pygeox_code_fixed}\n```"
            #assistant_content = f"<think>{pygeox_sft_thinking}</think> <answer>{pygeox_code_fixed}</answer>"
            
            # Create the message structure
            messages = [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": assistant_content}
            ]
            
            # Write as JSONL
            json_line = json.dumps({"messages": messages}, ensure_ascii=False)
            fout.write(json_line + '\n')
            processed_count += 1
            
        except Exception as e:
            print(f"Error processing {json_file}: {e}")
            skipped_count += 1

print(f"\nProcessing complete!")
print(f"Processed: {processed_count} files")
print(f"Skipped: {skipped_count} files")
print(f"Output written to: {output_file}")

Found 30225 JSON files in generated_data_no_penalty/json_processed
Found 46977 JSON files in generated_data_penalty/json_processed
Total: 46977 JSON files

Processing complete!
Processed: 46977 files
Skipped: 0 files
Output written to: sft_data_llm1.jsonl


# Generating new difficult data - with natural language 

In [5]:
import json
from pathlib import Path

# Load natural language descriptions from rewritten dataset
print("Loading natural descriptions from sft_balanced_8k_natural_test.jsonl...")
natural_descriptions = {}
with open("sft_balanced_8k_natural_test.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        if 'source_file' in data:
            source_file = data['source_file']
            # Extract user message (natural description)
            for msg in data['messages']:
                if msg['role'] == 'user':
                    natural_descriptions[source_file] = msg['content']
                    break

print(f"Loaded {len(natural_descriptions)} natural descriptions")

# Input directory
json_dir = Path("../generated_data_penalty/json_processed_fixed")

# Output file
output_file = "rl_data_hard_natural.jsonl"

# Statistics
stats = {"processed": 0, "skipped": 0, "missing_nl": 0}

# Find all JSON files (only hard difficulty - 3obj_)
json_files = sorted(json_dir.glob("3obj_*.json"))
print(f"Found {len(json_files)} hard difficulty JSON files in {json_dir}")

with open(output_file, 'w', encoding='utf-8') as fout:
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as fin:
                data = json.load(fin)
            
            # Get source file path
            source_file_path = str(json_file.resolve())
            
            # Check if we have a natural description for this file
            if source_file_path not in natural_descriptions:
                stats["missing_nl"] += 1
                continue
            
            # Extract required fields
            possible_solution = data['possible_solution']
            
            # Skip if required fields are missing
            if not possible_solution:
                stats["skipped"] += 1
                continue
            
            # Load system prompt
            with open("/mnt/disk0/r00922822/PyGeoX/model_training/system_prompt_rl.md", "r") as f: 
                system_prompt = f.read()

            # Use the natural description from rewritten dataset
            diagram_description = natural_descriptions[source_file_path]
            
            # Create expected format dictionaries
            expected_point_dict = {p: [None, None] for p in data["possible_solution"]["points"].keys()}
            expected_circle_dict = {c: None for c in data["possible_solution"]["circles"].keys()}

            # Create user prompt in the same format
            user_prompt = f"""Write python code to find the coordinates and circle radiuses for:
{diagram_description}

Required format of 'points' dictionary:
{json.dumps(expected_point_dict)}

Required format of 'circles' dictionary:
{json.dumps(expected_circle_dict)}"""

            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
            
            entry = {
                "messages": messages,
                "source_file": source_file_path,
                "difficulty": "hard"
            }
            
            # Write to output file
            json_line = json.dumps(entry, ensure_ascii=False)
            fout.write(json_line + '\n')
            stats["processed"] += 1
            
        except Exception as e:
            print(f"Error processing {json_file}: {e}")
            stats["skipped"] += 1

# Print statistics
print("\nProcessing complete!")
print("=" * 80)
print(f"Total processed: {stats['processed']} files")
print(f"Missing natural descriptions: {stats['missing_nl']} files")
print(f"Skipped (other reasons): {stats['skipped']} files")
print(f"\nOutput written to: {output_file}")
print("=" * 80)

Loading natural descriptions from sft_balanced_8k_natural_test.jsonl...
Loaded 22231 natural descriptions
Found 8202 hard difficulty JSON files in ../generated_data_penalty/json_processed_fixed

Processing complete!
Total processed: 7857 files
Missing natural descriptions: 345 files
Skipped (other reasons): 0 files

Output written to: rl_data_hard_natural.jsonl


# Prepare RL data for the second LLM

In [4]:
import json
from pathlib import Path

# Input directory
json_dir = Path("../generated_data_penalty/json_processed_fixed")

# Single output file
output_file = "rl_data_new.jsonl"

# Statistics
stats = {"processed": 0, "skipped": 0, "by_difficulty": {"easy": 0, "medium": 0, "hard": 0}}

# Find all JSON files
json_files = sorted(json_dir.glob("*.json"))
print(f"Found {len(json_files)} JSON files in {json_dir}")

with open(output_file, 'w', encoding='utf-8') as fout:
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as fin:
                data = json.load(fin)
            
            # Extract required fields
            diagram_description = data['nl_description']
            possible_solution = data['possible_solution']
            
            # Skip if required fields are missing
            if not diagram_description or not possible_solution:
                stats["skipped"] += 1
                continue
            
            # Determine difficulty based on filename
            difficulty_level = None
            if "1obj_" in json_file.name:
                difficulty_level = "easy"
            elif "2obj_" in json_file.name:
                difficulty_level = "medium"
            elif "3obj_" in json_file.name:
                difficulty_level = "hard"
            
            if not difficulty_level:
                stats["skipped"] += 1
                continue  # Skip files that don't match the pattern
            
            with open("/mnt/disk0/r00922822/PyGeoX/model_training/system_prompt_rl.md", "r") as f: system_prompt = f.read()

            diagram_description = data["nl_description"]
            expected_point_dict = {p: [None, None] for p in data["possible_solution"]["points"].keys()}
            expected_circle_dict = {c: None for c in data["possible_solution"]["circles"].keys()}

            user_prompt = f"""Write python code to find the coordinates and circle radiuses for:
{diagram_description}

Required format of 'points' dictionary:
{json.dumps(expected_point_dict)}

Required format of 'circles' dictionary:
{json.dumps(expected_circle_dict)}"""

            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]

            # Create entry with original file location and difficulty
            source_file_path = str(json_file.resolve())
            
            entry = {
                "messages": messages,
                "source_file": source_file_path,
                "difficulty": difficulty_level
            }
            
            # Write to output file
            json_line = json.dumps(entry, ensure_ascii=False)
            fout.write(json_line + '\n')
            stats["processed"] += 1
            stats["by_difficulty"][difficulty_level] += 1
            
        except Exception as e:
            print(f"Error processing {json_file}: {e}")
            stats["skipped"] += 1

# Print statistics
print("\nProcessing complete!")
print("=" * 80)
print(f"Total processed: {stats['processed']} files")
print(f"Total skipped: {stats['skipped']} files")
print(f"\nBy difficulty:")
for level, count in stats["by_difficulty"].items():
    print(f"  {level}: {count} files")
print(f"\nOutput written to: {output_file}")
print("=" * 80)

Found 46977 JSON files in ../generated_data_penalty/json_processed_fixed

Processing complete!
Total processed: 46977 files
Total skipped: 0 files

By difficulty:
  easy: 23840 files
  medium: 14935 files
  hard: 8202 files

Output written to: rl_data_new.jsonl


# OLD - PREPARE SFT DATA FOR THE SECOND LLM

In [2]:
import json
from pathlib import Path
import random

# Process json_processed_2 directory
json_dir_penalty = Path("../generated_data_penalty/json_processed_2")
json_files = sorted(json_dir_penalty.glob("*.json"))
random.shuffle(json_files)
output_file = "sft_data_llm2_with_system_prompt.jsonl"

print(f"Total: {len(json_files)} JSON files")

processed_count = 0
skipped_count = 0
entries_written = 0
r1_seen = 0

obj2_count = 0
obj3_count = 0
with open(output_file, 'w', encoding='utf-8') as fout:
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as fin:
                data = json.load(fin)
            
            # Only pick cases where the nl_description starts with "2obj_" or  "3obj_"
            if "2obj_" not in json_file.name and "3obj_" not in json_file.name:
                skipped_count += 1
                continue

            if "2obj_" in json_file.name:
                obj2_count +=1
            if "3obj_" in json_file.name:
                obj3_count +=1
            
            if obj2_count > 1050:
                continue
            if obj3_count > 1050:
                continue

            
            diagram_description = data["nl_description"]

            format = {
                "points": {point: [None,None] for point in list(data["possible_solution"]["points"].keys())},
                "circles": {circle: None for circle in list(data["possible_solution"]["circles"].keys())}
            }

            system_prompt = fr"""Please solve the user request, by first thinking, then answering, and finally verifying the answer.
Follow the following template:
<think> Thinking step by step <\think>
<answer> Final Answer </answer> 
<verify> Verify if answer is correct and finish by Writing Score: [0 or 1] </verify>"""

            user_prompt = f"""Your goal is to find the coordinates and circle radiuses of all points and circles in the following diagram:
{diagram_description}

The output format is a dictionary with the point coordinates and the circle radiusues (key is the name of the circle center):
```json
{json.dumps(format, indent=2)}
```"""

            # Process constructive_approach_data
            constructive_data = data.get('constructive_approach_data', {})
            if constructive_data:
                think = constructive_data.get('think')
                answer = constructive_data.get('answer')
                verify = constructive_data.get('verify')

                
                if think and answer and verify:
                    # Construct assistant message with format: <think>...</think> <answer>...</answer> <response>...</response>
                    assistant_content = f"<think>We will try a constructive approach to solve the problem.\n {think}</think> <answer>{answer}</answer> <verify>{verify}</verify>"
                    
                    # Create the message structure
                    messages = [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                        {"role": "assistant", "content": assistant_content}
                    ]
                    
                    # Write as JSONL
                    json_line = json.dumps({"messages": messages}, ensure_ascii=False)
                    fout.write(json_line + '\n')
                    entries_written += 1
            
            # Process code_approach_data
            code_data = data.get('code_approach_data', {})
            if code_data:
                think = code_data.get('think')
                answer = code_data.get('answer')
                verify = code_data.get('verify')
                
                if think and answer and verify:
                    # Construct assistant message with format: <think>...</think> <answer>...</answer> <response>...</response>
                    assistant_content = f"<think>We will try converting the problem into a system of equations and then optimize it numerically.\n {think}</think> <answer>{answer}</answer> <verify>{verify}</verify>"
                    
                    # Create the message structure
                    messages = [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                        {"role": "assistant", "content": assistant_content}
                    ]
                    
                    # Write as JSONL
                    json_line = json.dumps({"messages": messages}, ensure_ascii=False)
                    fout.write(json_line + '\n')
                    entries_written += 1
                else:
                    print(f"Skipping {json_file} because it doesn't have all required fields")
                    print(think, answer, verify)
            
            # Process r1_approach_data
            r1_data = False #data.get('r1_approach_data', {}) or data.get('R1_approach', {})
            if r1_data:
                think = r1_data.get('think', '')
                answer = r1_data.get('answer', '')
                verify = r1_data.get('verify', '')
                r1_seen += 1
                
                if think and answer and verify:
                    # Construct assistant message with format: <think>...</think> <answer>...</answer> <response>...</response>
                    assistant_content = f"<think>{think}</think> <answer>{answer}</answer> <verify>{verify}</verify>"
                    
                    # Create the message structure
                    messages = [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                        {"role": "assistant", "content": assistant_content}
                    ]
                    
                    # Write as JSONL
                    json_line = json.dumps({"messages": messages}, ensure_ascii=False)
                    fout.write(json_line + '\n')
                    entries_written += 1
                else:
                    print(f"Skipping {json_file} R1 approach because it doesn't have all required fields")
                    print(think, answer, verify)
            
            # Only count as processed if at least one entry was written
            if constructive_data or code_data or r1_data:
                processed_count += 1
            else:
                skipped_count += 1
            
        except Exception as e:
            print(f"Error processing {json_file}: {e}")
            skipped_count += 1

print(f"\nProcessing complete!")
print(f"Processed: {processed_count} files")
print(f"Skipped: {skipped_count} files")
print(f"Entries written: {entries_written}")
print(f"Output written to: {output_file}")

print(f"R1 data: {r1_seen}")

Total: 12751 JSON files

Processing complete!
Processed: 1592 files
Skipped: 6559 files
Entries written: 3184
Output written to: sft_data_llm2_with_system_prompt.jsonl
R1 data: 0


# OLD - Prepare RL data for the second LLM

In [3]:
# Prepare RL data: Create a single output file with difficulty field (easy, medium, hard)
# Read from generated_data_penalty/json_processed_2_fixed
# Extract nl_description and possible_solution
# Format: user message with description + "Please generate a json code block with all the point coordinates and circle radiuses."
# Assistant message: possible_solution formatted as JSON code block
# Save original file location and difficulty level

import json
from pathlib import Path

# Input directory
json_dir = Path("../generated_data_penalty/json_processed_fixed")

# Single output file
output_file = "rl_data.jsonl"

# Statistics
stats = {"processed": 0, "skipped": 0, "by_difficulty": {"easy": 0, "medium": 0, "hard": 0}}

# Find all JSON files
json_files = sorted(json_dir.glob("*.json"))
print(f"Found {len(json_files)} JSON files in {json_dir}")

with open(output_file, 'w', encoding='utf-8') as fout:
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as fin:
                data = json.load(fin)
            
            # Extract required fields
            diagram_description = data['nl_description']
            possible_solution = data['possible_solution']
            
            # Skip if required fields are missing
            if not diagram_description or not possible_solution:
                stats["skipped"] += 1
                continue
            
            # Determine difficulty based on filename
            difficulty_level = None
            if "1obj_" in json_file.name:
                difficulty_level = "easy"
            elif "2obj_" in json_file.name:
                difficulty_level = "medium"
            elif "3obj_" in json_file.name:
                difficulty_level = "hard"
            
            if not difficulty_level:
                stats["skipped"] += 1
                continue  # Skip files that don't match the pattern
            

            format = {
                "points": {point: [None,None] for point in list(data["possible_solution"]["points"].keys())},
                "circles": {circle: None for circle in list(data["possible_solution"]["circles"].keys())}
            }

            system_prompt = f"""Please solve the user request, by first thinking, then answering, and finally verifying the answer.
Follow the following template:
<think> Thinking step by step <\think>
<answer> Final Answer </answer> 
<verify> Verify if answer is correct and finish by Writing Score: [0 or 1] </verify>"""

            user_prompt = f"""Your goal is to find the coordinates and circle radiuses of all points and circles in the following diagram:
{diagram_description}

The output format is a dictionary with the point coordinates and the circle radiusues (key is the name of the circle center):
```json
{json.dumps(format, indent=2)}
```"""

            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]

            # Create entry with original file location and difficulty
            source_file_path = str(json_file.resolve())
            
            entry = {
                "messages": messages,
                "source_file": source_file_path,
                "difficulty": difficulty_level
            }
            
            # Write to output file
            json_line = json.dumps(entry, ensure_ascii=False)
            fout.write(json_line + '\n')
            stats["processed"] += 1
            stats["by_difficulty"][difficulty_level] += 1
            
        except Exception as e:
            print(f"Error processing {json_file}: {e}")
            stats["skipped"] += 1

# Print statistics
print("\nProcessing complete!")
print("=" * 80)
print(f"Total processed: {stats['processed']} files")
print(f"Total skipped: {stats['skipped']} files")
print(f"\nBy difficulty:")
for level, count in stats["by_difficulty"].items():
    print(f"  {level}: {count} files")
print(f"\nOutput written to: {output_file}")
print("=" * 80)

Found 46977 JSON files in ../generated_data_penalty/json_processed_fixed

Processing complete!
Total processed: 46977 files
Total skipped: 0 files

By difficulty:
  easy: 23840 files
  medium: 14935 files
  hard: 8202 files

Output written to: rl_data.jsonl
